In [1]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from scipy.optimize import curve_fit
from statsmodels.tsa.ar_model import AutoReg, ar_select_order
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import scipy.stats as stats

In [2]:
def setup_api_client():
    """Set up and return OpenMeteo API client with caching and retry functionality"""
    
    cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    
    return openmeteo_requests.Client(session=retry_session)

In [3]:
def fetch_weather_data(client, latitude, longitude, start_date, end_date, timezone="Europe/Amsterdam"):
    """Fetch weather data from OpenMeteo API"""
    
    url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
    
    params = {'latitude': latitude, 'longitude': longitude, 'start_date': start_date,
              'end_date': end_date,'hourly': "temperature_2m", 'daily': "temperature_2m_mean"}
    
    responses = client.weather_api(url, params=params)
    return responses[0]

In [4]:
def parse_daily_data(response):
    """Parse daily temperature data from API response"""
    daily = response.Daily()
    daily_temperature = daily.Variables(0).ValuesAsNumpy()

    start_time = pd.to_datetime(daily.Time(), unit='s', utc=True)
    end_time = pd.to_datetime(daily.TimeEnd(), unit='s', utc=True)
    interval = pd.Timedelta(seconds=daily.Interval())
    
    daily_data = pd.DataFrame({'temperature_2m_mean': daily_temperature}, index=pd.date_range(start=start_time, end=end_time, freq=interval, inclusive='left'))
    
    return daily_data

In [5]:
def clean_and_prepare_data(daily_data):
    """
    Clean and prepare data with additional features for analysis
    """

    daily_data['day'] = np.arange(len(daily_data))
 
    daily_data['month'] = daily_data.index.month
    daily_data['day_of_year'] = daily_data.index.dayofyear

    monthly_to_season = {1: 1, 2: 1, 3: 2, 4: 2, 5: 2, 6: 3, 7: 3, 8: 3, 9: 4, 10: 4, 11: 4, 12: 1}
    daily_data['season'] = daily_data.index.month.map(monthly_to_season).astype(int)
    
    return daily_data.dropna()

In [6]:
client = setup_api_client()
response = fetch_weather_data(client=client, latitude=52.37, longitude=4.89, start_date="2020-08-10",end_date="2024-08-23")

In [7]:
daily_data = parse_daily_data(response)
daily_data = clean_and_prepare_data(daily_data)

In [ ]:
print("\nTemperature data summary:")
print(daily_data['temperature_2m_mean'].describe())

#### Plotting Time Series

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(daily_data['temperature_2m_mean'])
plt.title('Temperature Time Series', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.ylabel('Temperature', fontsize=18)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.tight_layout()
plt.show()

plt.figure(figsize=(15, 5))
plt.plot(daily_data['temperature_2m_mean'].rolling(window=7).mean())
plt.title('7-Day Rolling Mean', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.ylabel('Temperature', fontsize=18)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.tight_layout()
plt.show()


plt.figure(figsize=(15, 5))
plt.plot(daily_data['temperature_2m_mean'].rolling(window=7).var())
plt.title('7-Day Rolling Variance', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.ylabel('Variance', fontsize=18)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
decomposition = sm.tsa.seasonal_decompose(daily_data['temperature_2m_mean'], period=365)

# Plot Observed
plt.figure(figsize=(15, 4))
plt.plot(decomposition.observed)
plt.title('Observed', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.ylabel('Temperature', fontsize=18)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.tight_layout()
plt.show()

# Plot Trend
plt.figure(figsize=(15, 4))
plt.plot(decomposition.trend)
plt.title('Trend', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.ylabel('Trend', fontsize=18)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.tight_layout()
plt.show()

# Plot Seasonal
plt.figure(figsize=(15, 4))
plt.plot(decomposition.seasonal)
plt.title('Seasonal', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.ylabel('Seasonality', fontsize=18)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.tight_layout()
plt.show()

# Plot Residual
plt.figure(figsize=(15, 4))
plt.plot(decomposition.resid)
plt.title('Residual', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.ylabel('Residuals', fontsize=18)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.tight_layout()
plt.show()

#### Residual Analysis

In [ ]:
residuals = decomposition.resid.dropna()
plt.figure(figsize=(8, 6))
plt.hist(residuals, bins=30, density=True)
plt.title('Residuals Histogram', fontsize=20)
plt.xlabel('Residuals', fontsize=18)
plt.ylabel('Density', fontsize=18)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
stats.probplot(residuals, dist="norm", plot=plt)
plt.title('Q-Q Plot', fontsize=20)
plt.xlabel('Theoretical Quantiles', fontsize=18)
plt.ylabel('Sample Quantiles', fontsize=18)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
plot_acf(residuals, lags=40, ax=ax)
ax.set_title('Autocorrelation Function (ACF)', fontsize=20)
ax.set_xlabel('Lags', fontsize=18)
ax.set_ylabel('Autocorrelation', fontsize=18)
ax.tick_params(axis='both', labelsize=16)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
plot_pacf(residuals, lags=40, ax=ax)
ax.set_title('Partial Autocorrelation Function (PACF)', fontsize=20)
ax.set_xlabel('Lags', fontsize=18)
ax.set_ylabel('Partial Autocorrelation', fontsize=18)
ax.tick_params(axis='both', labelsize=16)
plt.tight_layout()
plt.show()


In [ ]:
from statsmodels.tsa.stattools import adfuller


adf_result = adfuller(residuals)


adf_statistic = adf_result[0]
p_value = adf_result[1]
n_lags = adf_result[2]
n_obs = adf_result[3]
critical_values = adf_result[4]


print("\nAugmented Dickey-Fuller Test Results")

print(f"ADF Statistic       : {adf_statistic:.4f}")
print(f"p-value             : {p_value:.4f}")
print(f"Number of Lags Used      : {n_lags}")
print(f"Number of Observations   : {n_obs}")
print("Critical Values     :")
for key, value in critical_values.items():
    print(f"  {key:<10}: {value:.4f}")

if p_value < 0.05:
    print("The residuals are likely stationary (reject H₀ at 5% level).")
else:
    print("The residuals are likely non-stationary (fail to reject H₀ at 5% level).")


### Deterministic Model Fitting

In [ ]:
def deterministic_model(t, a, b, alpha, theta):
    """Model with linear trend and annual seasonality"""
    return a + b*t + alpha*np.sin(2*np.pi/365.25 * t + theta)

popt, pcov = curve_fit(deterministic_model, 
                       daily_data['day'].values, 
                       daily_data['temperature_2m_mean'],
                       p0=[10, 0.001, 5, 0])  # Initial guesses

a, b, alpha, theta = popt
print(f"Fitted parameters:\na (intercept): {a:.4f}\nb (trend): {b:.4f}\nalpha (amplitude): {alpha:.4f}\ntheta (phase): {theta:.4f}")


In [ ]:
time = daily_data['day'].values
model_fit = deterministic_model(time, a, b, alpha, theta)
daily_data['model_fit'] = model_fit
daily_data['residuals'] = daily_data['temperature_2m_mean'] - model_fit

plt.figure(figsize=(15, 6))
plt.plot(daily_data['day'], daily_data['residuals'], label='Residuals', color='green')
plt.axhline(y=0, color='red', linestyle='--', label='Zero Line')

plt.title('Residuals Plot', fontsize=20)
plt.xlabel('Day', fontsize=18)
plt.ylabel('Residuals (°C)', fontsize=18)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.legend(fontsize=16)
plt.grid(True)
plt.tight_layout()
plt.show()


#### AR Order

In [ ]:
selector = ar_select_order(daily_data['residuals'].dropna(), maxlag=20)
best_lag = selector.ar_lags  
print(f"Optimal AR Order (AIC): {best_lag}")


ar_model = AutoReg(daily_data['residuals'].dropna(), lags=best_lag).fit()
print(ar_model.summary())
gamma = ar_model.params['residuals.L1'] 


kappa = 1- gamma
print(f"Mean-reversion parameter (κ): {kappa:.4f}")

In [19]:
def fourier_volatility(t, a0, *coeffs):
    K = len(coeffs) // 2
    sigma = a0 * np.ones_like(t)
    for k in range(1, K + 1):
        a_k, b_k = coeffs[2*k-2], coeffs[2*k-1]
        sigma += a_k * np.cos(2 * np.pi * k * t / 365.25) + \
                 b_k * np.sin(2 * np.pi * k * t / 365.25)
    return sigma

In [ ]:
# Example: Fit with K=2 harmonics (annual + semi-annual)
K = 2
initial_guess = [np.std(residuals)] + [0.1] * (2 * K)  # [a0, a1, b1, a2, b2]
# Time index (0 to N-1)
t = np.arange(len(residuals))

# Target: squared residuals (proxy for variance)
target = residuals ** 2

# Fit Fourier series to squared residuals
popt, pcov = curve_fit(
    lambda t, a0, a1, b1, a2, b2: fourier_volatility(t, a0, a1, b1, a2, b2),
    t,
    target,
    p0=initial_guess
)

# Extract coefficients
a0, a1, b1, a2, b2 = popt
print(f"Fitted coefficients: a0={a0:.4f}, a1={a1:.4f}, b1={b1:.4f}, a2={a2:.4f}, b2={b2:.4f}")
# Estimated volatility series
sigma_t = np.sqrt(fourier_volatility(t, *popt))

# Add to DataFrame
daily_data['sigma_t'] = np.nan
daily_data.loc[residuals.index, 'sigma_t'] = sigma_t

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(daily_data.index, daily_data['sigma_t'], label='Estimated σ(t)', color='red')


plt.title('Seasonal Volatility Estimation via Fourier Series', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.ylabel('Volatility', fontsize=18)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.legend(fontsize=16)
plt.grid(True)
plt.tight_layout()
plt.show()


## Pricing Temperature Options

In [22]:
def simulate_temperature_paths(start_day, num_days, num_paths, deterministic_params, ar_params, volatility_params):
    """
    Simulate temperature paths using Euler discretization scheme.
    
    Parameters:
    -----------
    start_day : int
        Day index to start simulation from
    num_days : int
        Number of days to simulate
    num_paths : int
        Number of Monte Carlo paths to generate
    deterministic_params : tuple
        Parameters (a, b, alpha, theta) for deterministic model
    ar_params : dict
        AR model parameters including gamma (AR(1) coefficient)
    volatility_params : tuple
        Parameters for volatility model (a0, a1, b1, a2, b2)
    
    Returns:
    --------
    numpy.ndarray
        Array of shape (num_paths, num_days) containing simulated temperature paths
    """

    a, b, alpha, theta = deterministic_params
    gamma = ar_params['gamma']
    a0, a1, b1, a2, b2 = volatility_params

    paths = np.zeros((num_paths, num_days))
    last_residual = 0  

    for path in range(num_paths):
        residual = last_residual
        for i in range(num_days):
            day = start_day + i

            det_temp = deterministic_model(day, a, b, alpha, theta)
            vol = np.sqrt(fourier_volatility(day % 365, a0, a1, b1, a2, b2))

            epsilon = np.random.normal(0, 1)
            residual = gamma * residual + vol * epsilon
            paths[path, i] = det_temp + residual
    
    return paths
def calculate_degree_days(temperatures, reference_temp=18.0, mode='HDD'):
    """
    Calculate Heating Degree Days (HDD) or Cooling Degree Days (CDD).
    """

    if mode == 'HDD':
        return np.sum(np.maximum(reference_temp - temperatures, 0))
    elif mode == 'CDD':
        return np.sum(np.maximum(temperatures - reference_temp, 0))
    else:
        raise ValueError("Mode must be 'HDD' or 'CDD'")
    
def call_option_payoff(degree_days, strike, alpha=1.0, cap=float('inf')):
    """
    Calculate the payoff for a call option with cap.

    """
    return min(alpha * max(degree_days - strike, 0), cap)
def put_option_payoff(degree_days, strike, alpha=1.0, floor=float('inf')):
    """
    Calculate the payoff for a put option with floor.
    """
    return min(alpha * max(strike - degree_days, 0), floor)

def collar_option_payoff(degree_days, strike1, strike2, alpha=1.0, beta=1.0, cap=float('inf'), floor=float('inf')):
    """
    Calculate the payoff for a collar option.
    """
    call_payoff = min(alpha * max(degree_days - strike1, 0), cap)
    put_payoff = min(beta * max(strike2 - degree_days, 0), floor)
    return call_payoff - put_payoff


In [23]:
def price_option_mc(temperatures, strike, option_type='call', dd_type='HDD', 
                   alpha=1.0, cap=float('inf'), floor=float('inf'),
                   strike2=None, beta=1.0, r=0.02, T=0.0):
    """
    Method for pricing a weather derivative option using Monte Carlo simulation.
    
    Parameters:
    temperatures : numpy.ndarray
        Array of shape (num_paths, num_days) containing simulated temperature paths
    strike : float
        Strike price of the option
    option_type : str
        Type of option: 'call', 'put', or 'collar'
    dd_type : str
        Type of degree days: 'HDD' or 'CDD'
    alpha : float
        Notional multiplier (tick size)
    cap : float
        Maximum payoff cap
    floor : float
        Maximum payoff floor
    strike2 : float
        Second strike price (only for collar)
    beta : float
        Second notional multiplier (only for collar)
    r : float
        Risk-free interest rate
    T : float
        Time to maturity in years
    
    Returns:
    float
        Option price
    """
    num_paths = temperatures.shape[0]
    payoffs = np.zeros(num_paths)
    

    for i in range(num_paths):
        degree_days = calculate_degree_days(temperatures[i, :], mode=dd_type)
        
        if option_type == 'call':
            payoffs[i] = call_option_payoff(degree_days, strike, alpha, cap)
        elif option_type == 'put':
            payoffs[i] = put_option_payoff(degree_days, strike, alpha, floor)
        elif option_type == 'collar':
            if strike2 is None:
                raise ValueError("strike2 must be provided for collar options")
            payoffs[i] = collar_option_payoff(degree_days, strike, strike2, alpha, beta, cap, floor)
    
    option_price = np.exp(-r * T) * np.mean(payoffs)
    return option_price

In [ ]:
start_day = daily_data['day'].max() + 1  
num_days = 90  
num_paths = 20000  


deterministic_params = (a, b, alpha, theta)

ar_params = {'gamma': gamma}

volatility_params = (a0, a1, b1, a2, b2)


simulated_paths = simulate_temperature_paths(start_day, num_days, num_paths, deterministic_params, ar_params, volatility_params)

plt.figure(figsize=(15, 6))


for i in range(min(10, num_paths)):  
    plt.plot(np.arange(num_days), simulated_paths[i, :], color='blue', alpha=0.3)


plt.plot(np.arange(num_days), np.mean(simulated_paths, axis=0), color='red', linewidth=2, label='Mean Path')

plt.title(f'Monte Carlo Simulation: {num_paths} Temperature Paths', fontsize=20)
plt.xlabel('Days from Start', fontsize=18)
plt.ylabel('Temperature (°C)', fontsize=18)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.grid(True)
plt.legend(fontsize=16)
plt.tight_layout()
plt.show()


In [25]:
strike = 300  
alpha = 10.0  
cap = 5000.0 

In [ ]:
option_price = price_option_mc(
        simulated_paths, strike, option_type='call', dd_type='HDD',
        alpha=alpha, cap=cap, r=0.02, T=num_days/365.0
    )
    
print(f"Price of HDD Call Option (K={strike}, $\\alpha$={alpha}, Cap={cap}): {option_price:.2f}")


hdd_values = np.array([calculate_degree_days(simulated_paths[i, :], mode='HDD') for i in range(num_paths)])

print("HDD Statistics:")
print(f"Mean: {np.mean(hdd_values):.2f}")
print(f"Std Dev: {np.std(hdd_values):.2f}")
print(f"Min: {np.min(hdd_values):.2f}")
print(f"Max: {np.max(hdd_values):.2f}")

In [ ]:
plt.figure(figsize=(15, 6))
plt.hist(hdd_values, bins=30, density=True, alpha=0.7, color='skyblue', edgecolor='black')

plt.axvline(x=strike, color='red', linestyle='--', linewidth=2, label=f'Strike K = {strike}')

plt.title('Distribution of Simulated HDDs', fontsize=20)
plt.xlabel('Heating Degree Days (HDD)', fontsize=18)
plt.ylabel('Density', fontsize=18)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.legend(fontsize=16)
plt.grid(True)
plt.tight_layout()
plt.show()
